# KosenMap Website 取扱説明書 —— 目次

**この Website(公開ページ・管理画面・本番ホスト)を動かすための取扱説明書。**
読むだけでなく、**書いてあるコマンドをそのセルから実行できる。**

対象は本番 `ito4.jp`(`/opt/kosenmap`)と、この PC の `server/scripts`。
**繋ぐ利用者は 2 つ**(2026-09-18 に分けた。[12](12-hardening-2026-09-15.ipynb) §7-4 B): 配備・`%%host`・控えは **`kmops`**(鍵 `~\.ssh\km_ops`。docker あり・**sudo なし**)、`sudo` が要るセルは **`km`**(鍵 `~\.ssh\km_vps`。**docker なし**)。
Android アプリは別リポジトリなので、ここには入っていない。

| ノートブック | 何が書いてあるか |
|---|---|
| [00-start](00-start.ipynb) | **この取扱説明書の使い方。** 準備と、セルの型 |
| [01-daily-check](01-daily-check.ipynb) | **日々の確認。** 自己検査・ホストの様子・メールが届いているか |
| [02-deploy](02-deploy.ipynb) | **配備。** 下見・配備・後片付け・困ったとき |
| [03-backup](03-backup.ipynb) | **バックアップと復元。** 取る・開く・週次タスク・添付の復号・戻す |
| [04-host-jobs-and-mail](04-host-jobs-and-mail.ipynb) | **ホストの定期処理とメール。** cron の時刻表・届くメールの一覧・試しに送る |
| [05-containers](05-containers.ipynb) | **コンテナの更新と追加。** Logto の上げ方・固定・足し方 |
| [06-emergency](06-emergency.ipynb) | **もしものとき。** まず叩く1本と、症状別の見どころ |
| [07-map-qr](07-map-qr.ipynb) | **地図の配信と QR。** |
| [08-architecture](08-architecture.ipynb) | **どう出来ているか。** 構造・設定の読み方・落とし穴・設計の約束 |
| [09-new-host](09-new-host.ipynb) | **新しいホストを作る・移す。** VPS の構築から切り替えまで |
| [10-security-review-2026-09-14](10-security-review-2026-09-14.ipynb) | **診断の報告書(2026-09-14)。** 何を見つけて何を直したか・本番で実行する手順 |
| [11-getting-started](11-getting-started.ipynb) | **新しく使う人の入口。** 全体の図・部品の役割・管理画面・Android アプリ・用語集 |
| [12-hardening-2026-09-15](12-hardening-2026-09-15.ipynb) | **多層防御の底上げ(2026-09-15)。** 網の分割・read_only・Soketi の Node 24・Logto の DB 利用者・管理画面の別オリジン・本番への当て方 |
| [13-local-env](13-local-env.ipynb) | **ローカル環境(LAN の検証機)。** `.env` の `KM_ENV=local` で、Let's Encrypt を使わずに本番と同じ構成を立てる |
| [14-domain-ito4](14-domain-ito4.ipynb) | **ito4.jp に統一する(2026-09-17)。** 旧 ito8795.com は同時に手放す・audience も https://ito4.jp/api へ・管理画面は admin.ito4.jp・外部スキャンの指摘 |
| [15-staff-org](15-staff-org.ipynb) | **教職員の自動付与(2026-09-18〜)。** 学校ドメインの JIT で Logto の組織へ入れ、組織ロールで「地図の錠を通る・教職員氏名を見る・閲覧不可の地点を見る」を与える |

記録として残している文書:

| 文書 | 中身 |
|---|---|
| [status.md](status.md) | **いまどうなっているか。** 実測値・踏んだ罠・残作業 |
| [plan.md](plan.md) | **これから何をするか。** 優先順と、やらないと決めたこと |
| [../Old/docs/](../Old/docs/) | 以前の手順書(md)。**細部の経緯はここ**。ノートブックに移した内容の元 |

# 日々の確認と自己検査

**直す前・配備する前・「何か変だ」と思ったときに、上から流す。** ここにあるのはすべて 🟢(読むだけ)。

> 使い方は [00-start.ipynb](00-start.ipynb)。**まず下のセルを1回実行する。**

| 印 | 意味 |
|---|---|
| 🟢 | **読むだけ。** 何も変えない。迷ったらここから |
| 🟡 | **手元が変わる。** この PC にファイルを作る・登録する。本番には触れない |
| 🔴 | **本番が変わる。** 実行前に `yes` の入力を求める |
| 🔑 | **別の窓で開く。** sudo のパスワードなど対話が要るもの |

In [ ]:
# 最初に1回だけ実行する(%%ps / %%host / %%terminal が使えるようになる)
import sys, pathlib
for _d in (pathlib.Path.cwd(), pathlib.Path.cwd() / 'docs', pathlib.Path.cwd() / 'server' / 'docs'):
    if (_d / 'km_nb.py').exists():
        sys.path.insert(0, str(_d))
        break
import km_nb
km_nb.load()

## 1. 手元の検査

### 自己検査(check.php)

**DB も Logto も要らない。** PHP の純粋な関数と、スクリプト・compose・ノートブックの決まりごとを見る。
最後に `すべて通過 (N 件)` と出れば通っている。

🟢 **読むだけ** —— 何も変えません。

In [ ]:
%%ps
php ..\src\scripts\check.php

### 名前で絞る / 一覧を見る

節の名前は `--list` で出る。`recaptcha` は環境依存なので、名前を書いたときだけ走る。

🟢 **読むだけ** —— 何も変えません。

In [ ]:
%%ps
php ..\src\scripts\check.php --list
php ..\src\scripts\check.php notebooks backup-notice

### PHP の構文検査

🟢 **読むだけ** —— 何も変えません。

In [ ]:
%%ps
$errors = 0
Get-ChildItem ..\src -Recurse -Filter *.php | Where-Object { $_.FullName -notmatch '\\vendor\\' } | ForEach-Object {
    $out = php -l $_.FullName 2>&1
    if ($LASTEXITCODE -ne 0) { $errors++; $out }
}
"構文エラー: $errors 件"

### 配備の下見

何が転送され、何が除外されるかを出すだけ。**接続もしない。**

🟢 **読むだけ** —— 何も変えません。

In [ ]:
%%ps
.\deploy-to-host.ps1 -WhatIfOnly

## 2. ホストの様子

### コンテナ・ディスク・メモリ・再起動

**healthy は「中の HTTP が返る」ことしか言っていない。** 利用者の経路が通るかは画面で見る
([05-containers](05-containers.ipynb) の §7)。

🟢 **読むだけ** —— 何も変えません。

In [ ]:
%%host
uptime
df -h / | awk 'NR==2 {print "ディスク / : " $5 " 使用(残り " $4 ")"}'
free -h | awk 'NR==2 {print "メモリ     : " $3 " / " $2 " 使用"}'
if [ -f /var/run/reboot-required ]; then echo "★ 再起動が必要です(更新が効いていません)"; else echo "再起動は要りません"; fi
echo
docker ps --format '{{.Names}}\t{{.Status}}' | sort

### 定期処理の仕込み(cron)

1行目の版が **5**、持ち主が **root:root 644** であること。**root 所有でない cron.d のファイルを、cron は黙って無視する**
(2026-09-09 に実際に起きた)。時刻表の読み方は [04-host-jobs-and-mail](04-host-jobs-and-mail.ipynb)。

🟢 **読むだけ** —— 何も変えません。

In [ ]:
%%host
head -1 /etc/cron.d/kosenmap-updates
stat -c '%U:%G %a %n' /etc/cron.d/kosenmap-updates /var/log/kosenmap
echo
grep -v '^#' /etc/cron.d/kosenmap-updates | grep -v '^[[:space:]]*$'

### メールが届いているか(直近24時間)

**メールは送信側からは成功に見える。** 本当に相手へ渡ったかは、送信サーバーの記録の `status=sent` で見る。
宛先の名前の部分は伏せて出す。時刻は UTC。

🟢 **読むだけ** —— 何も変えません。

In [ ]:
%%host
docker logs -t --since 24h km-mailserver 2>&1 \
  | grep -E 'status=(sent|bounced|deferred)' \
  | sed -E 's/^([0-9-]+)T([0-9:]{8})[^ ]* .*to=<[^@>]*@([^>]*)>.*status=([a-z]+).*/\1 \2 UTC  …@\3  \4/'
echo "件数: $(docker logs --since 24h km-mailserver 2>&1 | grep -cE 'status=sent')"

## 3. 控え

### ホストの控え(新しい順)

🟢 **読むだけ** —— 何も変えません。

In [ ]:
%%host
ls -lt --time-style=+%Y-%m-%d_%H:%M backups | awk 'NR>1 {print $6, $5, $7}' | head -20

### 手元の控えと、週次タスク

🟢 **読むだけ** —— 何も変えません。

In [ ]:
%%ps
.\open-backup.ps1 -List
$task = Get-ScheduledTask -TaskName 'KosenMap バックアップ(週次)'
$i = $task | Get-ScheduledTaskInfo
"状態: $($task.State) / 前回: $($i.LastRunTime)(結果 $($i.LastTaskResult)) / 次回: $($i.NextRunTime)"
. .\backup-lib.ps1
$log = Join-Path (Get-KmBackupRoot) 'backup-task.log'
if (Test-Path $log) { Get-Content $log -Tail 20 -Encoding utf8 } else { "記録はまだありません: $log" }

## 4. 定期点検の目安

| いつ | 何を |
|---|---|
| **配備するたび** | `check.php` → 下見 → 配備 → 後片付けの「気になる点: 0」 → 画面で1周 |
| **週に1回** | 日曜の「バックアップを取りました」が届いたか。**届かない週は仕掛けが止まっている** |
| **月に1回** | 1日に「更新」「セキュリティ」の定期のお知らせが届いたか。**控えを実際に開いてみる**([03-backup](03-backup.ipynb)) |

細かい点検項目は [../Old/docs/vps-checklist.md](../Old/docs/vps-checklist.md) の「C. 定期点検」。